In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from abs_affinity_based_slotting.config import RAW_DIR, PROCESSED_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.slotting import build_full_instance
from abs_affinity_based_slotting.methods import CurrentSlotting, DemandGreedySlotting, LinearAssignmentSlotting
from abs_affinity_based_slotting.evaluation import Evaluator
import time

In [2]:
loader = WarehouseDataLoader(RAW_DIR)
dataset = loader.load_all()

split = split_picking_events(dataset.picking_events, test_size=0.2)
picking_train = split.train
picking_test = split.test
print(f"Train batches: {picking_train['batch_id'].nunique()}")
print(f"Test batches: {picking_test['batch_id'].nunique()}")

Train batches: 1600
Test batches: 400


In [ ]:
instance = build_full_instance(
    picking_train,
    dataset.initial_stock,
    dataset.distances,
    affinity_metric="jaccard"
)
print(f"Instance: {instance.n_skus} SKUs, {instance.n_locations} locations")

evaluator = Evaluator.from_tables(
    dataset.coordinates,
    dataset.distances,
    dataset.initial_stock
)

In [4]:
from dataclasses import replace

methods = {
    "current": CurrentSlotting(dataset.initial_stock),
    "demand_greedy": DemandGreedySlotting(),
    "linear_assignment": LinearAssignmentSlotting(),
}

results = {}
for name, method in methods.items():
    t0 = time.time()
    assignment = method.solve(instance)
    runtime = time.time() - t0
    
    metrics = evaluator.evaluate(assignment, picking_test)
    metrics = replace(metrics, runtime_seconds=runtime)
    results[name] = metrics
    print(f"{name}: {metrics.mean_batch_distance:.1f} mean distance, {runtime:.3f}s")

current: 52398.0 mean distance, 0.300s
demand_greedy: 26079.0 mean distance, 0.021s
linear_assignment: 26265.0 mean distance, 2338.200s


In [5]:
baseline_mean = results["current"].mean_batch_distance
comparison = []

for name, metrics in results.items():
    improvement = (baseline_mean - metrics.mean_batch_distance) / baseline_mean * 100
    comparison.append({
        "método": name,
        "distancia total": f"{metrics.total_distance:.0f}",
        "media/batch": f"{metrics.mean_batch_distance:.1f}",
        "mediana": f"{metrics.median_batch_distance:.1f}",
        "p95": f"{metrics.p95_batch_distance:.1f}",
        "mejora %": f"{improvement:+.1f}%",
        "tiempo (s)": f"{metrics.runtime_seconds:.3f}",
    })

pd.DataFrame(comparison)

,método,distancia total,media/batch,mediana,p95,mejora %,tiempo (s)
0,current,20959192,52398.0,52540.0,57233.2,+0.0%,0.300
1,demand_greedy,10431592,26079.0,25696.0,31320.4,+50.2%,0.021
2,linear_assignment,10506016,26265.0,26296.0,31438.0,+49.9%,2338.200


In [ ]:
from abs_affinity_based_slotting.slotting import slotting_cost

# Compute objective L (λ=1) for both solutions
for name in ["demand_greedy", "linear_assignment"]:
    assignment = results[name]  # Este es RouteMetrics, no Assignment
    # Necesito re-resolver para obtener Assignment
    
# Mejor: re-compute desde cero
assignment_greedy = DemandGreedySlotting().solve(instance)
assignment_linear = LinearAssignmentSlotting().solve(instance)

L_greedy = slotting_cost(assignment_greedy, instance, lam=1.0)
L_linear = slotting_cost(assignment_linear, instance, lam=1.0)

print(f"L (objective) - greedy: {L_greedy:.0f}")
print(f"L (objective) - linear: {L_linear:.0f}")
print(f"Difference: {L_linear - L_greedy:.0f} (linear should be ≤ greedy)")